In [1]:
from pathlib import Path
from typing import Callable
from converter import doc2text
from sentence_bert import make_embeddings as make_embeddings_SentenceBERT
from mistral import make_embeddings as make_embeddings_Mistral
from fast_text import make_embeddings as make_embeddings_FastText
from utils import split_markdown_by_headers
import numpy as np
import pickle

def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
  """Calcule la similarité cosinus entre deux vecteurs."""
  dot_product = np.dot(vec1, vec2)
  norm_vec1 = np.linalg.norm(vec1)
  norm_vec2 = np.linalg.norm(vec2)
  if norm_vec1 == 0 or norm_vec2 == 0:
    return 0 # Éviter la division par zéro
  return dot_product / (norm_vec1 * norm_vec2)


def make_embeddings(dossier: Path, embedding_function: Callable[[str], np.ndarray]):
    """
    Vectorise les textes en utilisant le modèle d'embedding fourni.

    embedding_function: fonction d'encodage qui prend un texte en entrée et retourne un embedding de type <numpy.ndarray>

    Retourne un dictionnaire :
    {
        "embedding": <numpy.ndarray>,
        "metadata": {
            "file": "Nom du fichier",
            "title": "Titre de la section"
        },
        "text": "Contenu du texte"
    }
    """

    embedding_docs = []

    # Embeddings des documents potentiels
    for element in dossier.rglob("*"):
        if not element.is_file():
            continue

        if not element.suffix.lower() == ".md":
            continue
        
        sections = split_markdown_by_headers(element)
        
        print(f"Vectorisation du fichier : {element.name} ({len(sections)} sections)", flush=True)
        for section in sections:
            embedding_docs.append({
                "embedding": embedding_function(section.get("content", "")),
                "text": section.get("content", ""),
                "metadata": {
                    "path": element.resolve(),
                    "file": section.get("filename", ""),
                    "title": section.get("title", ""),
                    #...
                }
            })

    return embedding_docs

def save_embeddings(embedding_docs: list, embedding_name: str):
    """
    Sauvegarde l'embedding des documents dans un fichier pickle.
    """
    with open(embedding_name, "wb") as f:
        pickle.dump(embedding_docs, f)

def load_embeddings(embedding_name: str)-> list:
    """
    Charge l'embedding des documents depuis un fichier pickle.
    """
    with open(embedding_name, "rb") as f:
        return pickle.load(f)

#
# MAIN
#

path = Path("./inputs")

In [2]:

# convertie les documents en textes Markdown
doc2text(path)


Start scan : inputs
Fichier : budget_2024.csv (déjà traité)
Fichier : Acceuil_affichage_Mairie de Triffouillis sur Loire.pdf (déjà traité)
Fichier : logo_triffouillis.webp (déjà traité)
Fichier : voeux2025_trifouillis.wav (déjà traité)
Fichier : Demande d'information sur l'entretien de la voirie.docx (déjà traité)
Fichier : Demande de participation à la réunion municipale sur la revitalisation du centre-ville.docx (déjà traité)
Fichier : Signalement et demande d'intervention pour un éclairage public défectueux.docx (déjà traité)
Fichier : Festival des Rires et des Blagues.docx (déjà traité)
Fichier : Fête des Saveurs Insolites.docx (déjà traité)
Fichier : Journée de l’Insolence Créative.docx (déjà traité)
Fichier : Le Carnaval des Objets Oubliés.docx (déjà traité)
Fichier : marche_noel_trifouillis_2023.png (déjà traité)
Fichier : Nuit des Arts Urbains.docx (déjà traité)
Fichier : Bulletin Municipal – Intervention Technique et Sécurité Routière.pdf (déjà traité)
Fichier : Bulletin Munic

In [3]:

# convertie les textes Markdown en emmbeddings
if not Path("SentenceBERT.embeddings").exists():
    embedding_docs = make_embeddings(path, make_embeddings_SentenceBERT)
    save_embeddings(embedding_docs, "SentenceBERT.embeddings")

if not Path("Mistral.embeddings").exists():
    embedding_docs = make_embeddings(path, make_embeddings_Mistral)
    save_embeddings(embedding_docs, "Mistral.embeddings")

if not Path("FastText.embeddings").exists():
    embedding_docs = make_embeddings(path, make_embeddings_FastText)
    save_embeddings(embedding_docs, "FastText.embeddings")


In [ ]:

# Mesurer la similarité
# en utilisant la similarité cosinus sur 2 vecteurs A et B nous pouvons déterminer à quel point ils sont similaires.
# La valeur de similarité cosinus varie entre -1 et 1
# 1 signifie que les vecteurs sont identiques
# 0 signifie qu'ils sont orthogonaux (aucune similarité)
# -1 signifie qu'ils sont opposés.
#    
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

embedding_docs = load_embeddings("SentenceBERT.embeddings")

vectors = np.array([
    item["embedding"]
    for item in embedding_docs
])

similarity_matrix = cosine_similarity(vectors)

similarity_matrix.shape

(217, 217)

In [ ]:
import pandas as pd

labels = [
    f"{item['metadata']['file']} - {item['metadata']['title']}"
    for item in embedding_docs
]


similarity_df = pd.DataFrame(
    similarity_matrix,
    index=labels,
    columns=labels
)

print(similarity_df.round(2))

                                                   Chers  \
                                             1.00   0.34   
Chers                                        0.34   1.00   
II. Horaires d'Ouverture et Contacts         0.42   0.61   
1. Accueil et Administration                 0.27   0.49   
2. Équipes Techniques et Voirie              0.31   0.65   
...                                           ...    ...   
2. Avancement des Travaux                    0.22   0.49   
3. Suivi Budgétaire                          0.48   0.39   
4. Prochains Jalons Prévisionnels            0.21   0.38   
5. Points de Vigilance / Risques Identifiés  0.24   0.44   
6. Conclusion                                0.41   0.67   

                                             II. Horaires d'Ouverture et Contacts  \
                                                                             0.42   
Chers                                                                        0.61   
II. Horaires d'Ouverture